## Notebook 3B - SVD / Matrix Factorization Model

### Goal
Use compatibility scores in `target.csv` as a user-user interaction matrix to build a recommendation system.

### Approach
This notebook focuses on building a compatibility matrix from the given data and applying Matrix Factorization, specifically Singular Value Decomposition (SVD), to learn latent embeddings for users and reconstruct predicted compatibility scores.

### Explanations

1.  **Why Matrix Factorization Makes Sense for This Task**
    Matrix factorization is a powerful technique for recommendation systems. In this context, we have a matrix where rows and columns represent users, and the values represent their pairwise compatibility scores. This matrix can be very large and potentially sparse (if not all user pairs have explicit compatibility scores). Matrix factorization helps us to discover underlying patterns and relationships between users, even when explicit data is limited. By decomposing the large matrix into smaller, low-rank matrices, we can capture the most important features that influence compatibility, which is crucial for making accurate predictions.

2.  **What Latent Factors Represent**
    Latent factors are abstract, unobservable features that the matrix factorization algorithm learns to represent the characteristics of users. For example, if we were predicting movie ratings, latent factors might represent genres (action, comedy, drama) or specific actors/directors. In our user-user compatibility scenario, latent factors could represent various aspects of user preferences, behavioral traits, or demographic similarities that contribute to their compatibility score. Each user is then represented as a vector in this latent factor space, where the values indicate how strongly they align with each latent factor.

3.  **Why SVD is Practical and Scalable**
    Singular Value Decomposition (SVD) is a widely used and robust matrix factorization technique. It decomposes a matrix into three simpler matrices: two orthogonal matrices and one diagonal matrix containing singular values. SVD is practical because it provides an optimal low-rank approximation of the original matrix, effectively reducing dimensionality while retaining the most significant information. It's also scalable in that efficient algorithms exist for computing SVD on large matrices, and its parallelizable nature makes it suitable for big data applications. While directly computing SVD on very large sparse matrices can be challenging, variants like Truncated SVD are designed to handle such cases efficiently by only computing the most significant singular values and vectors.

4.  **How Predictions Are Generated from the Reconstructed Matrix**
    After decomposing the original compatibility matrix `R` into `U`, `S`, and `Vt` matrices, we select a reduced number of `K` latent factors. We then reconstruct an approximation of the original matrix, often denoted as `R_hat`, by multiplying the truncated `U_k`, `S_k`, and `Vt_k` matrices (`R_hat = U_k @ S_k @ Vt_k`). This `R_hat` matrix contains the predicted compatibility scores for all user pairs. For any given `(src_user_idx, dst_user_idx)`, the value at `R_hat[src_user_idx, dst_user_idx]` represents the model's predicted compatibility score between those two users. These scores are typically clipped to be within a valid range (e.g., 0 to 1) if the original scores were bounded.

5.  **Files Saved**
    The following files are saved for future use and to allow the prediction model to be loaded without re-training:
    *   `svd_user2idx.pkl`: A Python dictionary mapping user IDs to their corresponding integer indices in the compatibility matrix.
    *   `svd_idx2user.pkl`: A Python dictionary mapping integer indices back to user IDs.
    *   `svd_pred_matrix.pkl`: The reconstructed compatibility matrix (`R_hat`), containing the predicted compatibility scores for all user pairs.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib


In [ ]:
target = pd.read_csv("target.csv")

print(target.head())
print("Rows:", len(target))
print("Unique users:", len(pd.unique(target[['src_user_id','dst_user_id']].values.ravel())))


   src_user_id  dst_user_id  compatibility_score
0         5001         5001               0.0000
1         5001         5002               0.1333
2         5001         5003               0.0714
3         5001         5004               0.0000
4         5001         5005               0.0833
Rows: 360000
Unique users: 600


In [ ]:
all_users = pd.unique(target[['src_user_id','dst_user_id']].values.ravel())

user2idx = {u:i for i,u in enumerate(all_users)}
idx2user = {i:u for u,i in user2idx.items()}

target["src_idx"] = target["src_user_id"].map(user2idx)
target["dst_idx"] = target["dst_user_id"].map(user2idx)

n_users = len(all_users)
print("Total users:", n_users)


Total users: 600


In [ ]:
# Create matrix (n_users x n_users)
R = np.zeros((n_users, n_users), dtype=np.float32)

for row in tqdm(target.itertuples(), total=len(target)):
    R[row.src_idx, row.dst_idx] = row.compatibility_score


100%|██████████| 360000/360000 [00:00<00:00, 374920.65it/s]


In [ ]:
from numpy.linalg import svd

# Rank (latent dimension) - tune this
K = 40

U, S, Vt = svd(R, full_matrices=False)

U_k = U[:, :K]
S_k = np.diag(S[:K])
Vt_k = Vt[:K, :]

R_hat = U_k @ S_k @ Vt_k

# Clip scores
R_hat = np.clip(R_hat, 0, 1)

print("Done. Pred matrix shape:", R_hat.shape)


Done. Pred matrix shape: (600, 600)


In [ ]:
joblib.dump(user2idx, "svd_user2idx.pkl")
joblib.dump(idx2user, "svd_idx2user.pkl")
joblib.dump(R_hat, "svd_pred_matrix.pkl")

print("Saved: svd_user2idx.pkl, svd_idx2user.pkl, svd_pred_matrix.pkl")


Saved: svd_user2idx.pkl, svd_idx2user.pkl, svd_pred_matrix.pkl


In [ ]:
def predict_pair(src_id, dst_id):
    if src_id not in user2idx or dst_id not in user2idx:
        return 0.5
    return float(R_hat[user2idx[src_id], user2idx[dst_id]])

print(predict_pair(target.loc[0,"src_user_id"], target.loc[0,"dst_user_id"]))


0.5031446814537048
